In [0]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit, to_date

#User pass date
try:
	arrival_date=dbutils.widgets.get("arrival_date")
except Exception:
	arrival_date=datetime.now().strftime("%Y-%m-%d")

#User pass catalog
try:
	catalog=dbutils.widgets.get("catalog")
except Exception:
	catalog="travel_bookings"

#User pass schema
try:
	schema=dbutils.widgets.get("schema")
except Exception:
	schema="default"

#User pass volume
try:
	base_volume=dbutils.widgets.get("volume")
except Exception:
	base_volume=f"/Volumes/{catalog}/{schema}/data"

In [0]:
#Read booking file and write into bronze layer table
booking_path=f"{base_volume}/booking_data/bookings_{arrival_date}.csv"

df=spark.read.csv(booking_path, inferSchema=True, header=True, escape='"', multiLine=True)
new_df=df.withColumn("ingestion_time", current_timestamp()).withColumn("business_date", to_date(lit("arrival_date")))

spark.sql(f"create schema if not exists {catalog}.bronze")

new_df.write.mode("append").format("delta").saveAsTable(f"{catalog}.bronze.booking_inc")

print(f"Ingested rows: {new_df.count()}")